In [52]:
import pandas as pd
from pyprojroot import here

In [53]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"

In [54]:
df = pd.read_csv(RAW_DATA_DIR/"crime/Municipal-Delitos-2015-2025_abr2026.csv", encoding='latin-1')

In [55]:
df['CVEGEO'] = df['Cve. Municipio'].astype(str).str.zfill(5)
df['crime_count'] = df.iloc[:, 9:21].sum(axis=1)
df = df.rename(columns={'Entidad': 'state', 'Municipio': 'municipality', 'Año': 'year'})

In [56]:
df['cvemun'] = df['Cve. Municipio'] - df['Clave_Ent'] * 1000
rows_to_drop = df[df['cvemun'] > 600].index
df = df.drop(index=rows_to_drop)

In [57]:
df['CVEGEO'].unique()

array(['01001', '01002', '01003', ..., '24059', '25019', '25020'],
      shape=(2478,), dtype=object)

In [58]:
A_crime = [ # Criminal activity associated with organized crime by the Federal Law Against Organized Crime
    'Secuestro',
    'Trata de personas',
    'Narcomenudeo',
    'Falsificación',
    'Corrupción de menores'
]

In [59]:
crime_df = df[df['Tipo de delito'].isin(A_crime)].copy()
A_crime_df = crime_df.groupby(['year', 'CVEGEO', 'state', 'municipality'])['crime_count'].sum().drop(columns='Unnamed: 0').reset_index()

In [60]:
munis_2025 = df[df['year'] == 2025][['CVEGEO']].drop_duplicates()

years = df['year'].unique()
skeleton = munis_2025.assign(key=1).merge(
    pd.DataFrame({'year': years, 'key': 1}), on='key'
).drop(columns='key')

ACdf_fixed = skeleton.merge(A_crime_df, on=['CVEGEO', 'year'], how='left')
ACdf_fixed['crime_count'] = ACdf_fixed['crime_count'].fillna(0).astype(int)
ACdf_fixed = ACdf_fixed.sort_values(['year', 'CVEGEO']).reset_index(drop=True)

In [61]:
ACdf_fixed.to_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding="latin-1")

ACdf_fixed.sample(5)

,CVEGEO,year,state,municipality,crime_count
16792,26015,2021,Sonora,Bavispe,1
922,17015,2015,Morelos,Miacatlán,8
450,12075,2015,Guerrero,Eduardo Neri,2
13537,20133,2020,Oaxaca,San Esteban Atatlahuca,0
15617,15080,2021,México,Sultepec,2
